# `n_color_immerse()` — coloring a 3D nematic director

This notebook explains why `n_color_immerse()` is constructed from an immersion of the real projective plane, and how the particular immersion used by Nematics3D was selected.

## 1. Nematic symmetry

A three-dimensional director is represented by a unit vector

$$
\mathbf n=(n_x,n_y,n_z),\qquad |\mathbf n|=1.
$$

For an ordinary polar vector, $\mathbf n$ and $-\mathbf n$ describe different states. A nematic director is different: it has no head and no tail. Therefore

$$
\boxed{\mathbf n\sim-\mathbf n.}
$$

Any physically meaningful coloring must respect this symmetry:

$$
\boxed{\mathbf c(\mathbf n)=\mathbf c(-\mathbf n).}
$$

The unit vectors themselves lie on the sphere $\mathbb S^2$, but antipodal points represent the same nematic orientation. The physical orientation space is therefore

$$
\mathbb S^2/\{\mathbf n\sim-\mathbf n\}\cong\mathbb{RP}^2,
$$

the real projective plane. Thus the actual problem is to assign colors to points of $\mathbb{RP}^2$.

## 2. What should a coloring scheme satisfy?

Before discussing any particular formula, consider the two most basic requirements for a useful coloring.

- **Uniqueness.** Different physical orientations should have different colors:

  $$
  [\mathbf n_1]\neq[\mathbf n_2]\quad\Longrightarrow\quad\mathbf c([\mathbf n_1])\neq\mathbf c([\mathbf n_2]).
  $$

  If uniqueness holds, a color identifies one and only one nematic orientation.

- **Continuity.** Nearby orientations should have nearby colors:

  $$
  [\mathbf n_2]\to[\mathbf n_1]\quad\Longrightarrow\quad\mathbf c([\mathbf n_2])\to\mathbf c([\mathbf n_1]).
  $$

  Otherwise a smooth director field could acquire artificial color discontinuities.

Together with the requirement that the two-dimensional orientation space remain locally a two-dimensional surface in color space, these are precisely the properties we want from an **embedding**.

## 3. The ideal coloring would be an embedding

A color can be represented by three coordinates, for example $(R,G,B)$. Ignoring the finite sRGB gamut for the moment, color space can therefore be regarded as a region of $\mathbb R^3$.

If we could assign colors continuously and uniquely while preserving the local structure of the nematic orientation space, we would have an embedding

$$
\boxed{\mathbf c:\mathbb{RP}^2\hookrightarrow\mathbb R^3.}
$$

Geometrically, an embedding would place $\mathbb{RP}^2$ inside three-dimensional color space as a surface without tearing, gluing, or self-intersection. Every orientation would have its own color, and continuously changing the orientation would continuously change the color.

Unfortunately, this ideal construction is mathematically impossible.

## 4. Why the ideal coloring is impossible

A classical topological result states that the real projective plane cannot be embedded in three-dimensional Euclidean space:

$$
\boxed{\mathbb{RP}^2\not\hookrightarrow\mathbb R^3.}
$$

This is not a limitation of RGB, a bad choice of formula, or an insufficient numerical optimizer. It is a property of the topology of $\mathbb{RP}^2$.

Therefore a continuous, globally unique three-dimensional coloring of nematic orientations does not exist. Restricting the target further to the sRGB cube clearly cannot remove this obstruction.

We must relax one of the ideal requirements. For visualization, continuity is particularly important: artificial color jumps in a smooth director field are highly undesirable. We therefore retain continuity and local distinguishability, but give up **global uniqueness**.

This changes the mathematical problem from finding an embedding to finding an **immersion**.

## 5. Relaxing embedding to immersion

An immersion

$$
\mathbf c:\mathbb{RP}^2\looparrowright\mathbb R^3
$$

is locally surface-like but is allowed to intersect itself globally. More precisely, its differential has rank two everywhere:

$$
\operatorname{rank}(D_T\mathbf c)=2.
$$

So the difference between the two concepts is simple:

- an **embedding** preserves local structure and is globally one-to-one;
- an **immersion** preserves local structure but may have global self-intersections.

For our coloring problem, a self-intersection means that two sufficiently different nematic orientations can occasionally have the same color. That ambiguity is unavoidable. Locally, however, small changes in the two independent orientation directions still produce distinguishable changes in color.

This is the compromise implemented by `n_color_immerse()`.

## 6. Boy's surface: an immersion of $\mathbb{RP}^2$

The standard example of an immersion of the real projective plane into $\mathbb R^3$ is **Boy's surface**. It realizes exactly the topology we need: it represents $\mathbb{RP}^2$ continuously and without local collapse, while allowing the global self-intersections that topology makes unavoidable.

Nematics3D uses a polynomial Boy-type immersion

$$
\mathbf p_B(\mathbf n)=\bigl(p_1(\mathbf n),p_2(\mathbf n),p_3(\mathbf n)\bigr),
$$

where

$$
\begin{aligned}
p_1&=\frac12\left[(2x^2-y^2-z^2)+2yz(y^2-z^2)+zx(x^2-z^2)+xy(y^2-x^2)\right],\\[4pt]
p_2&=\frac78\left[(y^2-z^2)+zx(z^2-x^2)+xy(y^2-x^2)\right],\\[4pt]
p_3&=\frac18(x+y+z)\left[(x+y+z)^3+4(y-x)(z-y)(x-z)\right].
\end{aligned}
$$

It automatically respects nematic symmetry:

$$
\mathbf p_B(\mathbf n)=\mathbf p_B(-\mathbf n).
$$

But a mathematical Boy surface is not yet a good colormap. Its coordinates have no intrinsic relation to human color perception, the Cartesian axes need not have intuitive colors, and an arbitrary placement may extend outside the displayable sRGB gamut.

The remaining problem is therefore not *whether* to use an immersion. It is: **which placement of a Boy-type immersion in color space gives the most useful scientific colormap?**

## 7. Searching for an optimal Boy-surface colormap

We keep the Boy-type immersion fixed and apply an affine transformation

$$
\boxed{\mathbf c_{\rm sRGB}(\mathbf n)=A\mathbf p_B(\mathbf n)+\mathbf b.}
$$

The matrix $A$ can rotate, stretch, and shear the immersed surface, while $\mathbf b$ translates it. For nonsingular $A$, these operations preserve the immersion property.

This turns the design problem into an optimization problem over $A$ and $\mathbf b$. The topology is already correct; we now need utility functions that tell us which placement is visually preferable.

The criteria used below are deliberately introduced only at this stage. They do not determine the topological construction. They select the most useful color realization among the admissible Boy-type immersions.

## 8. Perceptual color space: OKLab

The final colors are stored as sRGB values, but Euclidean distance in encoded RGB is not a good approximation to perceived color difference. We therefore evaluate the utility functions in **OKLab**.

Let

$$
f(\mathbf n)=\operatorname{OKLab}(\mathbf c_{\rm sRGB}(\mathbf n))=(L,a,b).
$$

The chroma is

$$
C=\sqrt{a^2+b^2}.
$$

Large $C$ corresponds to a more chromatic color, whereas small $C$ lies closer to the neutral gray axis.

## 9. Utility functions and constraints

Once the topology has been fixed by the immersion, we judge candidate affine transformations using three main visual objectives and one hard display constraint.

### 9.1 Local metric fidelity

A useful colormap should not make one infinitesimal orientation direction extremely visible while making another nearly invisible. Restrict the derivative of $f$ to the two-dimensional tangent plane and define

$$
G=(D_Tf)^T(D_Tf).
$$

The local distortion score is

$$
\boxed{J_{\rm loc}=\frac{\langle\operatorname{tr}(G^2)\rangle}{\langle\operatorname{tr}G\rangle^2}-\frac12.}
$$

Smaller $J_{\rm loc}$ means that local angular changes are represented more uniformly in perceptual color space. The normalization removes the trivial possibility of improving the score merely by globally scaling the color surface.

### 9.2 Semantic axis colors

For intuitive spatial interpretation, we would like

$$
\mathbf e_x\rightarrow\text{red},\qquad\mathbf e_y\rightarrow\text{green},\qquad\mathbf e_z\rightarrow\text{blue}.
$$

Axis errors are measured as OKLab distances from the exact sRGB primaries. In the final optimization, the maximum allowed error for each axis is calibrated from the previous production map:

$$
\delta_{\rm axis}=0.051845.
$$

### 9.3 Vividness

A map can satisfy the first two criteria and still devote too much orientation space to gray or muddy colors. We therefore explicitly reward OKLab chroma, using the mean chroma

$$
\langle C\rangle
$$

as the vividness objective.

### 9.4 sRGB gamut

Every generated color must satisfy

$$
0\le R,G,B\le1.
$$

This is treated as a hard constraint rather than repaired afterward by clipping. Post-hoc clipping can collapse distinct colors onto the gamut boundary and alter the local geometry that was optimized.

## 10. From the first optimization to the selected map

The first optimization considered local metric fidelity and axis fidelity, but did not explicitly reward vividness. Along that trade-off frontier, a knee appeared near

$$
J_{\rm loc}\approx0.43.
$$

That solution was a reasonable compromise for the objectives that had been included. Visual inspection, however, showed that too much of the orientation space remained at low chroma. This is an important optimization lesson: a Pareto-optimal solution can only optimize the quantities that were actually put into the problem.

The revised optimization therefore asks

$$
\boxed{\max_{A,\mathbf b}\;\langle C\rangle}
$$

subject to

$$
J_{\rm loc}\le t,
$$

the calibrated red/green/blue axis tolerances, and the global sRGB gamut constraint.

The parameter $t$ now has a clear interpretation: it controls how much local perceptual distortion we are willing to tolerate in exchange for greater vividness. The candidates $t=0.43$, $0.55$, and $0.70$ illustrate this trade-off. The selected production map uses

$$
\boxed{t=0.55,}
$$

which provides a substantial chroma improvement over the stricter $0.43$ solution without accepting the larger local distortion allowed by $0.70$. Note that the old $0.43$ result and the final $0.55$ result arise from different optimization formulations, because vividness was added explicitly in the revised problem.

## 11. Numerical gamut verification

The local metric objective was evaluated using deterministic Fibonacci sampling of orientation space. Gamut constraints were imposed on a denser directional set. Because a finite sample cannot by itself prove that the entire continuous immersed surface lies inside the RGB cube, the optimization used an active-set refinement procedure:

1. optimize using the current sampled gamut constraints;
2. evaluate the candidate on a much denser set of directions;
3. add any directions that violate the gamut to the constraint set;
4. re-optimize;
5. repeat until dense verification finds no excursion outside $[0,1]^3$.

The local metric quadrature used 1200 deterministic Fibonacci directions, while final candidate verification used $10^5$ directions.

## 12. Final production map

For the selected $J_{\rm loc}\le0.55$ solution, Nematics3D uses

$$
\mathbf c_{\rm sRGB}=A\mathbf p_B+\mathbf b,
$$

with

$$
A=\begin{pmatrix}
0.5022508927 & 0.0814191820 & 0.4278817283\\
-0.2622468169 & 0.4198664553 & 0.2843783906\\
-0.2603273419 & -0.3829942093 & 0.3705024139
\end{pmatrix},
$$

and

$$
\mathbf b=(0.3810134663,\;0.4051244319,\;0.4114207202).
$$

These numbers are not intended to have an independent physical interpretation. They are the optimized placement of the Boy-type immersion in the sRGB cube under the perceptual objectives and constraints described above.

## 13. Usage

`n_color_immerse()` accepts a normalized director or an array of normalized directors whose final axis has length three.

In [ ]:
import numpy as np
from nematics3d.field import n_color_immerse

n = np.eye(3)
colors = n_color_immerse(n)
colors

The nematic symmetry can be checked directly:

In [ ]:
np.allclose(n_color_immerse(n), n_color_immerse(-n))

## 14. How to interpret the colors

The essential point is that this colormap is an **immersion, not an embedding**. Nearby nematic orientations are represented continuously and remain locally distinguishable, but the color is not a globally unique coordinate for orientation. Two distant orientations can coincide in color because global self-intersection is topologically unavoidable in any immersion of $\mathbb{RP}^2$ into three dimensions.

The purpose of `n_color_immerse()` is therefore not to make RGB an invertible encoding of the director. It is to obtain a continuous, locally informative, perceptually useful visualization while respecting the fundamental nematic symmetry.